In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01030/BraTS2021_01030_t1ce.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01030/BraTS2021_01030_flair.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01030/BraTS2021_01030_t2.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01030/BraTS2021_01030_t1.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01030/BraTS2021_01030_seg.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01358/BraTS2021_01358_flair.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01358/BraTS2021_01358_t2.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01358/BraTS2021_01358_t1ce.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01358/BraTS2021_01358_seg.nii
/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset/BraTS2021_01358/BraTS2021_01358_t1.nii
/kaggle/

# Cell 1 — Install Dependencies

In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies                          ║
# ╚══════════════════════════════════════════════════════════╝
# In Kaggle: Add BraTS2021 dataset via Dataset tab first
# Search: "BraTS 2021 Training Data" and add to notebook
import subprocess, sys, os

def pip(cmd):
    r = subprocess.run(f'{sys.executable} -m pip {cmd} -q',
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'❌ pip {cmd}\n{r.stderr[-500:]}')
    return r.returncode == 0

print('📦 Installing stock nnU-Net + deps...')
pip('install nnunetv2')
pip('install nibabel')

import importlib, site
importlib.invalidate_caches()
for sp in site.getsitepackages():
    if sp not in sys.path:
        sys.path.insert(0, sp)

import nnunetv2
print(f'✅ nnunetv2 → {nnunetv2.__file__}')
assert 'MoME' not in nnunetv2.__file__, '❌ Wrong nnunetv2 loaded'
print('✅ Clean stock nnU-Net confirmed.')

📦 Installing stock nnU-Net + deps...
✅ nnunetv2 → /usr/local/lib/python3.12/dist-packages/nnunetv2/__init__.py
✅ Clean stock nnU-Net confirmed.


# Cell 2 — Configure Environment

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Configure nnU-Net Environment                 ║
# ╚══════════════════════════════════════════════════════════╝
import os

BASE         = '/kaggle/working/nnUNet_workdir'
RAW          = os.path.join(BASE, 'nnUNet_raw')
PREPROCESSED = os.path.join(BASE, 'nnUNet_preprocessed')
RESULTS      = os.path.join(BASE, 'nnUNet_results')

for d in [RAW, PREPROCESSED, RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = RAW
os.environ['nnUNet_preprocessed'] = PREPROCESSED
os.environ['nnUNet_results']      = RESULTS

print('✅ Environment configured:')
print(f'   RAW          → {RAW}')
print(f'   PREPROCESSED → {PREPROCESSED}')
print(f'   RESULTS      → {RESULTS}')

✅ Environment configured:
   RAW          → /kaggle/working/nnUNet_workdir/nnUNet_raw
   PREPROCESSED → /kaggle/working/nnUNet_workdir/nnUNet_preprocessed
   RESULTS      → /kaggle/working/nnUNet_workdir/nnUNet_results


# Cell 3 — Find BraTS Data & Convert to nnU-Net Format

In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Convert BraTS2021 (Kaggle nested structure)   ║
# ╚══════════════════════════════════════════════════════════╝
import os, shutil
import nibabel as nib
import numpy as np
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

SOURCE_DIR   = '/kaggle/input/datasets/vietanh21/brats-2021-task-1-dataset'
DATASET_ID   = 123
DATASET_NAME = f'Dataset{DATASET_ID:03d}_BraTS2021'
N_CASES      = 50    # increase for more; set to None for all

target_dir = os.path.join(os.environ['nnUNet_raw'], DATASET_NAME)
images_tr  = os.path.join(target_dir, 'imagesTr')
labels_tr  = os.path.join(target_dir, 'labelsTr')
os.makedirs(images_tr, exist_ok=True)
os.makedirs(labels_tr, exist_ok=True)

def find_nii_in(folder):
    """
    Given a folder, return the first .nii or .nii.gz file inside it.
    If the path itself is already a file, return it directly.
    Returns None if nothing found.
    """
    if os.path.isfile(folder):
        return folder
    if os.path.isdir(folder):
        for f in sorted(os.listdir(folder)):
            if f.endswith('.nii.gz') or f.endswith('.nii'):
                return os.path.join(folder, f)
    return None

def get_modality_file(case_path, case_id, modality):
    """
    Resolve actual .nii file for a given modality.
    Handles both:
      - case_path/CaseID_modality.nii/someactualfile.nii   (nested)
      - case_path/CaseID_modality.nii.gz                   (flat)
      - case_path/CaseID_modality.nii                      (flat)
    """
    # Try nested subfolder first (your dataset's pattern)
    subfolder = os.path.join(case_path, f'{case_id}_{modality}.nii')
    result = find_nii_in(subfolder)
    if result:
        return result

    # Try flat .nii.gz
    flat_gz = os.path.join(case_path, f'{case_id}_{modality}.nii.gz')
    if os.path.isfile(flat_gz):
        return flat_gz

    # Try flat .nii
    flat = os.path.join(case_path, f'{case_id}_{modality}.nii')
    if os.path.isfile(flat):
        return flat

    return None

# Collect all case IDs
all_cases = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
    and d.startswith('BraTS2021_')
])

if N_CASES is not None:
    all_cases = all_cases[:N_CASES]

assert len(all_cases) > 0, f'❌ No BraTS case folders found in {SOURCE_DIR}'
print(f'📂 Found {len(all_cases)} cases. Converting...')

MODALITY_MAP = {
    'flair': '0000',
    't1':    '0001',
    't1ce':  '0002',
    't2':    '0003',
}

copied, skipped, failed = 0, 0, []

for case_id in all_cases:
    case_path = os.path.join(SOURCE_DIR, case_id)
    case_ok   = True

    # ── Images ────────────────────────────────────────────────────────────
    for mod, suffix in MODALITY_MAP.items():
        dst = os.path.join(images_tr, f'{case_id}_{suffix}.nii.gz')
        if os.path.exists(dst):
            skipped += 1
            continue

        src = get_modality_file(case_path, case_id, mod)
        if src is None:
            print(f'⚠️  [{case_id}] Missing modality: {mod}')
            case_ok = False
            continue

        # Copy — convert .nii to .nii.gz if needed
        if src.endswith('.nii.gz'):
            shutil.copy2(src, dst)
        else:
            # Load and resave as .nii.gz
            img = nib.load(src)
            nib.save(img, dst)

    # ── Label ──────────────────────────────────────────────────────────────
    label_dst = os.path.join(labels_tr, f'{case_id}.nii.gz')
    if os.path.exists(label_dst):
        skipped += 1
    else:
        seg_src = get_modality_file(case_path, case_id, 'seg')
        if seg_src is None:
            print(f'⚠️  [{case_id}] Missing segmentation')
            case_ok = False
        else:
            img  = nib.load(seg_src)
            data = np.array(img.dataobj, dtype=np.float32)
            data[data == 4] = 3   # BraTS: 0,1,2,4 → 0,1,2,3
            nib.save(
                nib.Nifti1Image(data.astype(np.uint8), img.affine, img.header),
                label_dst
            )
            copied += 1

    if not case_ok:
        failed.append(case_id)

print(f'\n✅ Conversion done.')
print(f'   Copied  : {copied}')
print(f'   Skipped : {skipped}')
if failed:
    print(f'   ⚠️  Failed cases ({len(failed)}): {failed[:10]}')

# ── dataset.json ──────────────────────────────────────────────────────────
n_labels = len(os.listdir(labels_tr))
generate_dataset_json(
    target_dir,
    channel_names={'0': 'FLAIR', '1': 'T1', '2': 'T1ce', '3': 'T2'},
    labels={'background': 0, 'necrotic': 1, 'edema': 2, 'enhancing': 3},
    num_training_cases=n_labels,
    file_ending='.nii.gz',
    dataset_name=DATASET_NAME,
    reference='BraTS 2021',
    release='1.0',
    description='BraTS 2021 — labels remapped 0-3 for nnU-Net'
)
print(f'✅ dataset.json written. {n_labels} training cases registered.')

📂 Found 50 cases. Converting...

✅ Conversion done.
   Copied  : 50
   Skipped : 0
✅ dataset.json written. 50 training cases registered.


# Cell 4 — Verify Dataset

In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Verify Dataset Integrity                      ║
# ╚══════════════════════════════════════════════════════════╝
import os, json, nibabel as nib, numpy as np

DATASET_ID   = 123
DATASET_NAME = f'Dataset{DATASET_ID:03d}_BraTS2021'
target_dir   = os.path.join(os.environ['nnUNet_raw'], DATASET_NAME)
images_tr    = os.path.join(target_dir, 'imagesTr')
labels_tr    = os.path.join(target_dir, 'labelsTr')

with open(os.path.join(target_dir, 'dataset.json')) as f:
    meta = json.load(f)

print('📋 dataset.json:')
print(f"   name     : {meta['name']}")
print(f"   labels   : {meta['labels']}")
print(f"   n_train  : {meta['numTraining']}")
print(f"   channels : {meta['channel_names']}")
print(f'\n📁 imagesTr : {len(os.listdir(images_tr))} files')
print(f'📁 labelsTr : {len(os.listdir(labels_tr))} files')

label_files = sorted(os.listdir(labels_tr))
sample = nib.load(os.path.join(labels_tr, label_files[0]))
unique = np.unique(np.array(sample.dataobj, dtype=np.uint8)).tolist()
print(f'\n🔬 Label values in "{label_files[0]}": {unique}')
assert 4 not in unique, '❌ Label 4 still present!'
assert max(unique) <= 3,  '❌ Unexpected label values!'
print('✅ Labels correct (0–3). Ready for preprocessing.')

📋 dataset.json:
   name     : Dataset123_BraTS2021
   labels   : {'background': 0, 'necrotic': 1, 'edema': 2, 'enhancing': 3}
   n_train  : 50
   channels : {'0': 'FLAIR', '1': 'T1', '2': 'T1ce', '3': 'T2'}

📁 imagesTr : 200 files
📁 labelsTr : 50 files

🔬 Label values in "BraTS2021_00000.nii.gz": [0, 1, 2, 3]
✅ Labels correct (0–3). Ready for preprocessing.


# Cell 5 — Preprocess

In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — Plan & Preprocess                             ║
# ╚══════════════════════════════════════════════════════════╝
import subprocess, os

env = os.environ.copy()

result = subprocess.run(
    ['nnUNetv2_plan_and_preprocess', '-d', '123',
     '--verify_dataset_integrity', '-c', '3d_fullres'],
    env=env
)

print('✅ Preprocessing complete.' if result.returncode == 0
      else '❌ Preprocessing failed.')

Fingerprint extraction...
Dataset123_BraTS2021
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer


Extracting dataset fingerprint: 100%|██████████| 50/50 [00:32<00:00,  1.54it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139.5 177.  137.5], 3d_lowres: [140, 177, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([177. , 137.5]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data

Preprocessing cases: 100%|██████████| 50/50 [02:12<00:00,  2.65s/it]


✅ Preprocessing complete.


# Cell 6 — Train

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Train Standard nnU-Net on BraTS2021           ║
# ║  Live epoch output streamed to notebook                 ║
# ╚══════════════════════════════════════════════════════════╝
import os, subprocess, sys

os.environ['nnUNet_raw']          = '/kaggle/working/nnUNet_workdir/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/kaggle/working/nnUNet_workdir/nnUNet_preprocessed'
os.environ['nnUNet_results']      = '/kaggle/working/nnUNet_workdir/nnUNet_results'

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'   # force Python subprocess to not buffer output

cmd = [
    'nnUNetv2_train', '123', '3d_fullres', '0',
    '-tr', 'nnUNetTrainer',
    '--c',   # resume from checkpoint if one exists, safe to keep always on
]

print(f'🚀 Command: {" ".join(cmd)}\n')

process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,    # merge stderr into stdout
    text=True,
    bufsize=1,                   # line-buffered
)

# Stream every line as it arrives
epoch_count = 0
for line in process.stdout:
    line = line.rstrip()

    # Print everything
    print(line)

    # Extra marker when an epoch completes so it's easy to spot
    if 'train_loss' in line or 'Epoch' in line.lower():
        sys.stdout.flush()
    if 'Epoch' in line and '/' in line:
        epoch_count += 1

process.wait()

print(f'\n{"✅ Training complete." if process.returncode == 0 else "❌ Training failed."}')
print(f'   Epochs streamed: {epoch_count}')

🚀 Command: nnUNetv2_train 123 3d_fullres 0 -tr nnUNetTrainer --c


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-01 09:22:32.018417: Using torch.compile...
2026-05-01 09:22:33.628822: do_dummy_2d_data_aug: False
2026-05-01 09:22:33.629560: Using splits from existing split file: /kaggle/working/nnUNet_workdir/nnUNet_preprocessed/Datas